# 統計学特講

## 05 労働統計


### 目的

- 労働力調査の基本的な分類を確認する
- 完全失業率，労働力人口比率，就業率を計算する
- 完全失業率だけでは労働市場を十分に捉えられないことを確認する
- 未活用労働に関する指標を計算し，指標どうしの違いを比較する


### 注意点

このノートブックでは，人数の単位を **万人** として扱う．

実際の公表統計では，調査対象，季節調整の有無，原数値か季節調整値か，月次か四半期かによって値が異なることがある．また，人数は万人単位で公表されるため，細かく見ると構成項目を足した値と合計値が少しずれることがある．ここでは，労働市場の基本的な見方を理解することを目的とする．


### 前準備


In [ ]:
import pandas as pd              # データフレーム関連のおまじない
import matplotlib.pyplot as plt  # グラフ描画関連のおまじない

### 5-1 労働状態の分類


#### 労働状態の復習

労働力調査では，15歳以上人口を次のように分類する．

$$\text{15歳以上人口}=\text{労働力人口}+\text{非労働力人口}$$

$$\text{労働力人口}=\text{就業者}+\text{完全失業者}$$

ここで，就業者は実際に働いていた人だけでなく，仕事をもちながら調査期間中に仕事をしなかった休業者も含む．

$$\text{就業者}=\text{従業者}+\text{休業者}$$


#### 重要指標

労働市場の状態を見る基本的な指標は次の3つである．

$$\text{完全失業率}=\frac{\text{完全失業者}}{\text{労働力人口}}\times100$$

$$\text{労働力人口比率}=\frac{\text{労働力人口}}{\text{15歳以上人口}}\times100$$

$$\text{就業率}=\frac{\text{就業者}}{\text{15歳以上人口}}\times100$$


#### 労働統計データをデータフレームに読み込み


In [ ]:
data_url = "https://raw.githubusercontent.com/tnarumi/lecture-statistics/main/data/labor_force_japan.csv"
df_labor = pd.read_csv(data_url)
df_labor

このCSVファイルは，次の列からなる

```date```：年月，
```population_15plus```：15歳以上人口，
```at_work```：従業者，
```absent_from_work```：休業者, 
```complete_unemployed```：完全失業者

#### データの確認


##### データフレームの概要を確認


In [ ]:
df_labor.info()

##### 基本統計量の一覧表示


In [ ]:
df_labor.describe()

##### 日付データに変換


In [ ]:
df_labor["date"] = pd.to_datetime(df_labor["date"])
df_labor.head()

### 5-2 基本指標を計算する


#### 就業者の算出

就業者は従業者と休業者の合計である．

In [ ]:
df_labor["employed"] = df_labor["at_work"] + df_labor["absent_from_work"]
df_labor[["date", "employed", "at_work", "absent_from_work"]].head()

#### 労働力人口の算出

労働力人口は，就業者と完全失業者の合計である．


In [ ]:
df_labor["labor_force"] = df_labor["employed"] + df_labor["complete_unemployed"]
df_labor[["date", "labor_force", "employed", "complete_unemployed"]].head()

#### 非労働力人口の算出

非労働力人口は，15歳以上人口から労働力人口を引いたものである．ここでは，このCSVファイルの中の分類関係を使って計算する．

In [ ]:
df_labor["not_in_labor_force"] = df_labor["population_15plus"] - df_labor["labor_force"]
df_labor[["date", "not_in_labor_force", "population_15plus", "labor_force"]].head()

#### 完全失業率の計算

完全失業率は，労働力人口のうち，仕事がなく，仕事に就くことが可能で，求職活動をしている完全失業者がどれくらいいるかを表す．

$$\text{完全失業率}=\frac{\text{完全失業者}}{\text{労働力人口}}\times100$$


In [ ]:
df_labor["complete_unemployment_rate"] = df_labor["complete_unemployed"] / df_labor["labor_force"] * 100
df_labor[["date", "complete_unemployment_rate", "complete_unemployed", "labor_force"]].head()

#### 労働力人口比率の計算

労働力人口比率は，15歳以上人口全体のうち，どれくらいの人が労働市場に参加しているかを表す．

$$\text{労働力人口比率}=\frac{\text{労働力人口}}{\text{15歳以上人口}}\times100$$


In [ ]:
df_labor["labor_force_participation_rate"] = df_labor["labor_force"] / df_labor["population_15plus"] * 100
df_labor[["date", "labor_force_participation_rate", "labor_force", "population_15plus"]].head()

#### 就業率の計算

就業率は，15歳以上人口全体のうち，どれくらいの人が就業しているかを表す．

$$\text{就業率}=\frac{\text{就業者}}{\text{15歳以上人口}}\times100$$


In [ ]:
df_labor["employment_rate"] = df_labor["employed"] / df_labor["population_15plus"] * 100
df_labor[["date", "employment_rate", "employed", "population_15plus"]].head()

#### 基本指標の直近の値を確認


In [ ]:
df_labor[["date", "complete_unemployment_rate", "labor_force_participation_rate", "employment_rate"]].tail()

#### 基本指標を同じグラフに描く


In [ ]:
df_labor.plot(
    x="date",
    y=["complete_unemployment_rate", "labor_force_participation_rate", "employment_rate"],
    figsize=(10, 5),
    marker="o",
    grid=True
)

plt.xlabel("date")
plt.ylabel("rate (%)")
plt.title("Labor market indicators")
plt.legend(loc="best")
plt.show()

#### 考察

1. 完全失業率が高い時期はいつか．
2. 就業率は長期的に上昇しているか，それとも低下しているか．
3. 完全失業率と就業率は，常に反対方向に動いているか．

### 5-3 完全失業率だけを見ることの限界

#### 人数の動きを確認する

完全失業率が低下している場合でも，完全失業者が就業者に移ったのか，非労働力人口に移ったのかによって意味は異なる．そこで，人数の動きをあわせて確認する．


In [ ]:
df_labor.plot(
    x="date",
    y=["employed", "complete_unemployed", "not_in_labor_force"],
    figsize=(10, 5),
    marker="o",
    grid=True
)

plt.xlabel("date")
plt.ylabel("10 thousand persons")
plt.title("Employment status")
plt.legend(loc="best")
plt.show()


#### 変化量を計算する

`diff()` は，1つ前の行との差を計算する関数である．


In [ ]:
df_labor["complete_unemployment_rate_diff"] = df_labor["complete_unemployment_rate"].diff()
df_labor["labor_force_participation_rate_diff"] = df_labor["labor_force_participation_rate"].diff()
df_labor["employment_rate_diff"] = df_labor["employment_rate"].diff()

df_labor[[
    "date",
    "complete_unemployment_rate",
    "complete_unemployment_rate_diff",
    "labor_force_participation_rate",
    "labor_force_participation_rate_diff",
    "employment_rate",
    "employment_rate_diff"
]].head()

#### 完全失業率が低下した時期を確認する


In [ ]:
unemployment_rate_decreased = df_labor[df_labor["complete_unemployment_rate_diff"] < 0]

unemployment_rate_decreased[[
    "date",
    "complete_unemployment_rate_diff",
    "labor_force_participation_rate_diff",
    "employment_rate_diff"
]].head(10)


完全失業率が低下している時期に，労働力人口比率や就業率がどのように変化しているかを確認する．完全失業率が下がっていても，就業率が上がっていない場合には，労働市場の改善と単純には言えない可能性がある．


#### 考察

1. 完全失業率が低下している時期に，就業率も上昇しているか．
2. 完全失業率が低下している時期に，労働力人口比率が低下していることはあるか．
3. 完全失業率だけを見る場合と，就業率や労働力人口比率も見る場合で，解釈はどのように変わるか．


### 5-4 休業者を見る


#### 就業者に占める休業者の割合


In [ ]:
df_labor["absent_share"] = df_labor["absent_from_work"] / df_labor["employed"] * 100
df_labor[["date", "absent_share", "employed", "absent_from_work"]].head()

#### 休業者割合の時系列プロット


In [ ]:
df_labor.plot(x="date", y="absent_share", figsize=(10, 4), marker="o", grid=True)

plt.xlabel("date")
plt.ylabel("share (%)")
plt.title("Share of absent workers among employed persons")
plt.show()


休業者は就業者に含まれるため，休業者が増えても，完全失業率には直接反映されない．ただし，労働市場の状態を理解するうえでは重要な情報である．


### 5-5 未活用労働

#### 未活用労働とは

完全失業者は，仕事がなく，仕事に就くことが可能で，求職活動をしている人である．しかし，労働市場には，完全失業者以外にも，より多く働きたい人や，条件が整えば働きたい人がいる．

完全失業率に入らないが，労働市場に余力として存在する人たちを捉えるために，未活用労働に関する指標が用いられる．


#### 未活用労働データをデータフレームに読み込み

In [ ]:
data_lu = "https://raw.githubusercontent.com/tnarumi/lecture-statistics/main/data/labor_underutilization_japan.csv"
df_lu = pd.read_csv(data_lu)
df_lu

このCSVファイルは，次の列をもつ

```date```：四半期の開始月．たとえば，```2018-01``` は2018年1-3月期を表す，
```labor_force```：労働力人口，
```employed```：就業者，
```unemployed```：失業者，
```additional_work_wanted_employed```：追加就労希望就業者，
```potential_labor_force```：潜在労働力人口

```unemployed``` は，未活用労働指標で用いる詳細集計の失業者である．
ここでの「失業者」とは，現在就業しておらず，1か月以内に仕事を探していて，仕事があればすぐ仕事に就くことができる者をいい，従来の「完全失業者」の定義における求職活動期間「1週間以内」を「1か月以内」に拡張した者としている．

そのため，ここでの「労働力人口」は，就業者と失業者を合わせたものとしており，従来の，就業者と完全失業者を合わせたものとは異なる．

#### データの確認


In [ ]:
df_lu.info()

#### 日付データに変換


In [ ]:
df_lu["date"] = pd.to_datetime(df_lu["date"])
df_lu.head()

#### 未活用労働指標を計算する

ここでは，次の4つの指標を計算する．

$$LU1=\frac{\text{失業者}}{\text{労働力人口}}\times100$$

$$LU2=\frac{\text{失業者}+\text{追加就労希望就業者}}{\text{労働力人口}}\times100$$

$$LU3=\frac{\text{失業者}+\text{潜在労働力人口}}{\text{労働力人口}+\text{潜在労働力人口}}\times100$$

$$LU4=\frac{\text{失業者}+\text{追加就労希望就業者}+\text{潜在労働力人口}}{\text{労働力人口}+\text{潜在労働力人口}}\times100$$

LU1は失業者だけを分子にした指標である．LU4は，失業者だけでなく，追加就労希望就業者と潜在労働力人口も含めた広い指標である．


In [ ]:
df_lu["LU1"] = df_lu["unemployed"] / df_lu["labor_force"] * 100

df_lu["LU2"] = (
    (df_lu["unemployed"] + df_lu["additional_work_wanted_employed"])
    / df_lu["labor_force"]
    * 100
)

df_lu["LU3"] = (
    (df_lu["unemployed"] + df_lu["potential_labor_force"])
    / (df_lu["labor_force"] + df_lu["potential_labor_force"])
    * 100
)

df_lu["LU4"] = (
    (
        df_lu["unemployed"]
        + df_lu["additional_work_wanted_employed"]
        + df_lu["potential_labor_force"]
    )
    / (df_lu["labor_force"] + df_lu["potential_labor_force"])
    * 100
)

df_lu[["date", "LU1", "LU2", "LU3", "LU4"]].head()

#### 未活用労働指標を比較する


In [ ]:
df_lu.plot(x="date", y=["LU1", "LU2", "LU3", "LU4"], figsize=(10, 5), marker="o", grid=True)

plt.xlabel("date")
plt.ylabel("rate (%)")
plt.title("Labor underutilization indicators")
plt.legend(loc="best")
plt.show()

LU1だけを見ると，失業者として分類される人だけを見ていることになる．LU2，LU3，LU4を見ることで，より多く働きたい就業者や，条件が整えば働きたい非労働力人口も含めて，労働市場の余力を確認できる．


#### 未活用労働の内訳を見る


In [ ]:
df_lu.plot(
    x="date",
    y=["unemployed", "additional_work_wanted_employed", "potential_labor_force"],
    figsize=(10, 5),
    marker="o",
    grid=True
)

plt.xlabel("date")
plt.ylabel("10 thousand persons")
plt.title("Components of labor underutilization")
plt.legend(loc="best")
plt.show()

#### 考察

1. LU1とLU4の差が大きい時期はいつか．
2. LU4がLU1より大きくなるのはなぜか．
3. 完全失業率だけを見た場合と，未活用労働指標を見た場合で，労働市場の解釈はどのように変わるか．


### 5-6 練習用

課題：下記内容を確認しなさい．

1. 完全失業率が最も高い時期を確認しなさい．
2. 就業率が最も低い時期を確認しなさい．
3. 休業者割合が最も高い時期を確認し，その時期に何が起きていたかを考察しなさい．
4. LU1とLU4の差を計算し，差が大きい時期を確認しなさい．
5. 年齢別・男女別データを用意できる場合，性別・年齢階級別の就業率を比較しなさい．
6. 正規・非正規雇用のデータを用意できる場合，非正規雇用者数または非正規雇用比率の推移を確認しなさい．


In [ ]:
# ここにコードを各自で追加してください
